# 卷积神经网络

## 从全连接层到卷积

### 不变性

1. 平移不变性（translationinvariance）：不管检测对象出现在图像中的哪个位置，神经⽹络的前⾯⼏层应该对相同的图像区域具有相似的反应，即为“平移不变性”

2. 局部性（locality）：神经⽹络的前⾯⼏层应该只探索输⼊图像中的局部区域，⽽不过度在意图像中相隔较远区域的关系，这就是“局部性”原则。最终，可以聚合这些局部特征，以在整个图像级别进⾏预测。

### 多层感知机的限制

#### 局部性

以前，多层感知机可能需要数⼗亿个参数来表⽰⽹络中的⼀层，⽽现在卷积神经⽹络通常只需要⼏百个参数，⽽且不需要改变输⼊或隐藏表⽰的维数。参数⼤幅减少的代价是，我们的特征现在是平移不变的，并且当确定每个隐藏活性值时，每⼀层只包含局部的信息。以上所有的权重学习都将依赖于归纳偏置。当这种偏置与现实相符时，我们就能得到样本有效的模型，并且这些模型能很好地泛化到未知数据中。但如果这偏置与现实不符时，⽐如**当图像不满⾜平移不变时，我们的模型可能难以拟合我们的训练数据**。

### 练习

1.为什么平移不变性可能也不是好主意呢？

平移不变性可能会降低模型的准确性和泛化能力。对于某些任务，平移不变性并不是必须的特性。例如，对于图像分类任务，我们通常希望模型能够识别物体的位置和姿态，并根据这些信息对其进行分类。在这种情况下，平移不变性可能会降低模型的准确性，因为它忽略了物体的位置和姿态等重要信息。 其次，平移不变性可能会导致模型的泛化能力下降。

  参考：https://arxiv.org/pdf/1805.12177.pdf

2.描述一个类似的音频卷积层的架构。

一种基于卷积神经网络的音频特征生成方法，首先对声音信号进行预处理和离散傅里叶变换计算声音信号的幅度谱，形成二维谱图信号；然后搭建以上述二维谱图信号为输入的一维卷积神经网络并进行模型训练，得到特征生成器模型；最后对待测声音进行预处理和离散傅里叶变换得到二维谱图信号，并将其送入训练好的一维卷积神经网络，通过卷积网络计算，得到输出即为所要生成的音频特征，实现声音信号的音频特征生成。

3.卷积层也适合于文本数据吗？为什么？

卷积层也适合于文本数据。 在自然语言处理中，文本数据通常表示为词向量矩阵，其中每行代表一个词的向量表示。卷积层可以在这个矩阵上进行卷积操作，类似于图像卷积层中对图像进行卷积操作。 在卷积层中，卷积核会在输入矩阵上进行滑动窗口计算，输出一个新的特征矩阵。在文本数据中，这个特征矩阵可以看作是对输入文本的不同n-gram特征的提取。例如，一个大小为3的卷积核可以提取出输入文本中每个长度为3的n-gram特征。这些特征可以用于后续的分类或者回归任务。 此外，卷积层还可以与循环神经网络（RNN）结合使用，形成卷积神经网络（CNN）和循环神经网络（RNN）的混合模型。这种模型可以同时捕捉文本中的局部特征和全局特征，提高模型的性能。 因此，卷积层适用于文本数据，可以对文本数据进行卷积操作，提取出不同n-gram特征，并且可以与RNN结合使用，提高模型的性能。

## 图像卷积

### 互卷积运算

在⼆维互相关运算中，卷积窗⼝从输⼊张量的左上⻆开始，从左到右、从上到下滑动。当卷积窗⼝滑动到新⼀个位置时，包含在该窗⼝中的部分张量与卷积核张量进⾏按元素相乘，得到的张量再求和得到⼀个单⼀的标量值，由此我们得出了这⼀位置的输出张量值。

输出⼤⼩等于输⼊⼤⼩$n_h \times n_w$减去卷积核⼤⼩$k_h \times k_w$，即：
$$(n_h - k_h + 1) \times (n_w - k_w + 1).$$

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
def corr2d(X, K): #@save
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()

    return Y

In [3]:
X = torch.arange(9).reshape(3, 3)
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

### 卷积层

卷积层对输⼊和卷积核权重进⾏互相关运算，并在添加标量偏置之后产⽣输出。所以，卷积层中的两个被训练的参数是卷积核权重和标量偏置。就像我们之前随机初始化全连接层⼀样，在训练基于卷积层的模型时，我们也随机初始化卷积核权重。

In [9]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, X):
        return corr2d(X, self.weight) + self.bias

conv2d = Conv2D((2, 2))
conv2d(X)

tensor([[ 3.2260,  5.3204],
        [ 9.5091, 11.6034]], grad_fn=<AddBackward0>)

⾼度和宽度分别为和的卷积核可以被称为卷积或卷积核。我们也将带有卷积核的卷积层称为卷积层。

### 图像中目标的边缘检测

如下是卷积层的⼀个简单应⽤：通过找到像素变化的位置，来检测图像中不同颜⾊的边缘。⾸先，我们构造⼀个像素的⿊⽩图像。中间四列为⿊⾊（0），其余像素为⽩⾊（1）

In [11]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

接下来，我们构造⼀个⾼度为、宽度为的卷积核K。当进⾏互相关运算时，如果⽔平相邻的两元素相同，则输出为零，否则输出为⾮零。

In [12]:
K = torch.tensor([[1.0, -1.0]])

现在，我们对参数X（输⼊）和K（卷积核）执⾏互相关运算。如下所⽰，输出Y中的1代表从⽩⾊到⿊⾊的边缘，-1代表从⿊⾊到⽩⾊的边缘，其他情况的输出为0。

In [13]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

现在我们将输⼊的⼆维图像转置，再进⾏如上的互相关运算。其输出如下，之前检测到的垂直边缘消失了。不出所料，这个卷积核K只可以检测垂直边缘，⽆法检测⽔平边缘。

In [14]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

### 学习卷积核

In [16]:
# 构造一个二维卷积层，它具有1个输出通道和形状为(1, 2)的卷积核
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)

# 这个二维卷积层使用四维输入和输出格式(批量大小、通道、高度、宽度)
# 其中批量大小和通道都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # 迭代卷积核
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch: {i}, loss: {l.sum():.3f}')

epoch: 1, loss: 5.614
epoch: 3, loss: 1.669
epoch: 5, loss: 0.578
epoch: 7, loss: 0.219
epoch: 9, loss: 0.087


In [17]:
conv2d.weight.data.reshape(1, 2)

tensor([[ 0.9616, -1.0217]])

我们学习到的卷积核权重⾮常接近我们之前定义的卷积核K。

### 互相关和卷积

    为了得到正式的卷积运算输出，我们需要执⾏严格卷积运算，⽽不是互相关运算。幸运的是，它们差别不⼤，我们只需⽔平和垂直翻转⼆维卷积核张量，然后对输⼊张量执⾏互相关运算。
    值得注意的是，由于卷积核是从数据中学习到的，因此⽆论这些层执⾏严格的卷积运算还是互相关运算，卷积层的输出都不会受到影响。为了说明这⼀点，假设卷积层执⾏互相关运算并学习图6.2.1中的卷积核，该卷积核在这⾥由矩阵K表⽰。假设其他条件不变，当这个层执⾏严格的卷积时，学习的卷积核K′在⽔平和垂直翻转之后将与K相同。也就是说，当卷积层对图6.2.1中的输⼊和K′执⾏严格卷积运算时，将得到与互相关运算图6.2.1中相同的输出。

简单来说就是直接使用互相关运算即可，还能简化运算

### 特征映射和感受野

在卷积神经⽹络中，对于某⼀层的任意元素，其感受野（receptive field）是指在前向传播期间可能影响计算的所有元素（来⾃所有先前层）。\
请注意，感受野可能⼤于输⼊的实际⼤⼩(当输入为3x3时，通过两层卷积层，形状变化: $3 \times 3 \rightarrow 2 \times 2 \rightarrow 1 \times 1$，这里最后一层卷积层的输出2x2，但是由于输入的感受野是原本3x3提取过一次的，所以最后一层感受野是3x3的)。